# 개별종목 조합D — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합D 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합D의 피처 값만 지정합니다.
import json

COMBINATION = 'D'
FEATURE_COLUMNS = (
    'atr_ratio',
    'hv_20',
    'range_1',
    'range_20',
    'bb_bandwidth',
    'volume_z_20',
    'turnover_20',
    'log_amihud_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 24개 노트북이 각각 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20100331 ~ 20240822
학습 행·종목: 171557 162
조합D 피처: ('atr_ratio', 'hv_20', 'range_1', 'range_20', 'bb_bandwidth', 'volume_z_20', 'turnover_20', 'log_amihud_20')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130410,20130705,0.3681,0.3701,-0.0021,0.3530,0.2593,0.3190
1,2,balanced,999,20140414,20140711,0.4185,0.4741,-0.0556,0.3423,0.2107,0.2983
2,3,balanced,1248,20150421,20150716,0.3590,0.3330,0.0260,0.3582,0.3365,0.3509
3,4,balanced,1496,20160422,20160719,0.3884,0.4128,-0.0244,0.3596,0.2707,0.3315
4,5,NaN,1745,20170424,20170721,0.3978,0.4182,-0.0203,0.3352,0.2511,0.3165
5,6,balanced,1994,20180503,20180731,0.3655,0.3912,-0.0257,0.3580,0.3028,0.3397
6,7,balanced,2243,20190514,20190806,0.3993,0.4615,-0.0623,0.3579,0.2478,0.3214
7,8,balanced,2492,20200518,20200807,0.3579,0.3144,0.0434,0.3510,0.4437,0.3799
8,9,NaN,2741,20210518,20210810,0.3918,0.4423,-0.0505,0.3645,0.3296,0.3602
9,10,NaN,2989,20220519,20220812,0.3537,0.3343,0.0193,0.3489,0.2627,0.3158


,OOS 폴드 평균
accuracy,0.3792
training_majority_baseline_accuracy,0.3851
accuracy_minus_training_majority_baseline,-0.0059
macro_f1,0.3560
down_recall,0.3009
core_harmonic_mean,0.3385


재실행 명령: python scripts/run_stock_model_experiment.py
